# Notebook 03: Standard Machine Unlearning Baselines (`03_standard_mu.ipynb`)

### Objectives & Workflow:
1. **Load Exact Persisted Splits & Original Checkpoint**: Start every baseline from the exact pre-trained original model checkpoint $\theta_o$ across seeds `[0, 1, ..., 9]`.
2. **Execute Standard MU Baselines (Non-CMF)**:
   - **Random Label (RL)**
   - **NegGrad+ (Gradient Ascent-Descent with $L_2$ anchor)**
   - **SalUn (Saliency-Guided Parameter Updates)**
   - **SCRUB (Teacher-Student Distillation Divergence)**
   - **TarUn / UNSIR (Error-Maximizing Impair-Repair)**
3. **Compute 3-Tier Metrics & Illusion Gap**: Check Output Acc vs Linear Probe (LP) Acc and Nearest Class Center (NCC) Acc.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    evaluate_three_tier_metrics,
    visualize_feature_space_and_boundaries,
    MODEL_CONFIGS
)

from unlearning_methods import (
    unlearn_random_label,
    unlearn_neggrad_plus,
    unlearn_salun,
    unlearn_scrub,
    unlearn_tarun_unsir
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. Baseline Execution & Evaluation Loop (from `model_instruction.md`)

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

# Hyperparameter configurations matching model_instruction.md
cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

METHODS = ["random_label", "neggrad_plus", "salun", "scrub", "tarun_unsir"]

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
raw_train = datasets.CIFAR10(root="./data", train=True, download=True)
raw_test = datasets.CIFAR10(root="./data", train=False, download=True)

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)

# Search path in Kaggle input or local artifacts
kaggle_split = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = kaggle_split if os.path.exists(kaggle_split) else local_split
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split file not found at {kaggle_split} or {local_split}. Run 01_train_full.ipynb first.")
print(f"Using split artifact: {split_file}")
    
with open(split_file, "r") as f:
    split_info = json.load(f)

# Construct DataLoaders
train_r_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
train_f_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), train_tf)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

train_r_loader = DataLoader(train_r_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
train_f_loader = DataLoader(train_f_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

all_baseline_results = []

for seed in MODEL_SEEDS:
    kaggle_ckpt = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{seed}.pt"
    local_ckpt = f"./artifacts/checkpoints/original_{MODEL_NAME}_seed_{seed}.pt"
    ckpt_file = kaggle_ckpt if os.path.exists(kaggle_ckpt) else local_ckpt
    if not os.path.exists(ckpt_file):
        raise FileNotFoundError(f"Required original checkpoint not found at {kaggle_ckpt} or {local_ckpt}. Run 01_train_full.ipynb first.")
    print(f"Using original model checkpoint: {ckpt_file}")
        
    for method_name in METHODS:
        print(f"\n==================================================")
        print(f"--- Running MU Baseline: {method_name} | Model: {MODEL_NAME} | Seed: {seed} ---")
        print(f"==================================================")
        
        # Load original checkpoint
        model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
        model.load_state_dict(torch.load(ckpt_file, map_location=device))
        
        start_time = time.time()
        
        # Run Baseline Method
        if method_name == "random_label":
            unlearned_model = unlearn_random_label(model, train_r_loader, train_f_loader, num_classes=NUM_CLASSES, lr=1e-4, epochs=3, device=device)
        elif method_name == "neggrad_plus":
            unlearned_model = unlearn_neggrad_plus(model, train_r_loader, train_f_loader, alpha=0.95, lr=1e-4, epochs=3, device=device)
        elif method_name == "salun":
            unlearned_model = unlearn_salun(model, train_r_loader, train_f_loader, num_classes=NUM_CLASSES, threshold=0.5, lr=1e-4, epochs=3, device=device)
        elif method_name == "scrub":
            unlearned_model = unlearn_scrub(model, train_r_loader, train_f_loader, epochs=3, msteps=2, lr=1e-4, device=device)
        elif method_name == "tarun_unsir":
            unlearned_model = unlearn_tarun_unsir(model, train_r_loader, forget_loader=train_f_loader, num_classes=NUM_CLASSES, device=device)
            
        runtime = time.time() - start_time
        
        # Save unlearned checkpoint
        unlearn_ckpt_path = f"./artifacts/checkpoints/baseline_{method_name}_{MODEL_NAME}_seed_{seed}.pt"
        torch.save(unlearned_model.state_dict(), unlearn_ckpt_path)
        
        # Evaluate 3-tier metrics
        val_metrics = evaluate_three_tier_metrics(
            model=unlearned_model, train_loader=train_r_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
        )
        test_metrics = evaluate_three_tier_metrics(
            model=unlearned_model, train_loader=train_r_loader, retain_eval_loader=test_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
        )
        
        res = {
            "method": method_name,
            "model_name": MODEL_NAME,
            "seed": seed,
            "split_seed": SPLIT_SEED,
            "runtime_sec": runtime,
            "validation_metrics": val_metrics,
            "test_metrics": test_metrics
        }
        all_baseline_results.append(res)
        
        res_path = f"./artifacts/metrics/baseline_{method_name}_{MODEL_NAME}_seed_{seed}.json"
        with open(res_path, "w") as f:
            json.dump(res, f, indent=2)
            
        print(f"[{method_name}] Test Output Retain Acc: {test_metrics['output_retain']:.4f} | Forget Acc: {test_metrics['output_forget']:.4f}")
        print(f"[{method_name}] Test LP Retain Acc:     {test_metrics['lp_retain']:.4f} | Forget Acc: {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
        print(f"[{method_name}] Test NCC Retain Acc:    {test_metrics['ncc_retain']:.4f} | Forget Acc: {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
        print(f"[{method_name}] Runtime: {runtime:.2f}s | Saved to {res_path}")
        
        # Feature space & decision boundary visualization for first seed
        if seed == MODEL_SEEDS[0]:
            vis_save = f"./artifacts/plots/features_baseline_{method_name}_{MODEL_NAME}_seed_{seed}.png"
            visualize_feature_space_and_boundaries(
                model=unlearned_model,
                retain_loader=test_loader,
                forget_loader=forget_eval_loader,
                num_classes=NUM_CLASSES,
                forget_class=0,
                title=f"Feature Space & Linear Output: {method_name} ({MODEL_NAME})",
                device=device,
                save_path=vis_save
            )